# E-Commerce Sales Analysis

**A beginner-friendly data analytics project using Python and Jupyter Notebook**

## 1. Project introduction

An online shop records every purchase. These records can help a business understand what it sells, when customers buy, and which markets generate sales.

In this project, I analyse transaction data from a UK-based online retailer. I will use a simple workflow: **load the data → understand it → clean it → analyse it → create charts → explain the findings**.

### Business questions

1. How much revenue was recorded each month?
2. Which products generated the most revenue?
3. Which countries generated the most revenue?
4. Which customers had the highest recorded spend?
5. What was the average order value?

**Important:** This is a sales analysis, not a profit analysis. The dataset does not contain product costs.

## 2. Import the libraries

A library is a collection of ready-made tools. We use:
- **pandas** to work with tables (like Excel).
- **matplotlib** and **seaborn** to create charts.
- **numpy** for basic numerical work.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 3. Load the dataset

The data comes from the UCI Machine Learning Repository. The first time this cell runs, it downloads the dataset using the UCI Python package.

The original dataset is not modified. We create a working copy in the next steps.

In [ ]:
# Install the package once if it is not already installed:
# %pip install ucimlrepo openpyxl

from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)

## 4. Understand the data

Before changing anything, we should look at the data. This is called **exploratory data analysis (EDA)**.

- `head()` shows the first five rows.
- `shape` tells us how many rows and columns we have.
- `info()` shows column names and data types.
- `describe()` gives basic statistics for numerical columns.

In [ ]:
df.head()

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

### What do the columns mean?

| Column | Meaning |
|---|---|
| InvoiceNo | Invoice / order reference |
| StockCode | Product code |
| Description | Product name |
| Quantity | Number of units on the invoice line |
| InvoiceDate | Date and time of the transaction |
| UnitPrice | Price per unit in GBP (£) |
| CustomerID | Customer identifier |
| Country | Customer's country |

Each row represents a product line on an invoice. One invoice can contain multiple rows because an order can contain multiple products.

## 5. Check data quality

Real-world data is not always perfect. We check missing values and duplicate rows before doing analysis.

In [ ]:
# Count missing values in each column
df.isnull().sum()

In [ ]:
# Count exact duplicate rows
df.duplicated().sum()

## 6. Clean the data

We will make a separate copy so the original `df` remains unchanged.

Some records are cancellations. In this dataset, cancellation invoice numbers start with the letter **C**. Negative quantities can also represent returned goods. For this first project, our main sales analysis will focus on completed sales: non-cancelled invoices with positive quantities and positive unit prices.

We will keep rows with missing CustomerID for overall revenue analysis. We only need CustomerID when analysing individual customers.

In [ ]:
sales = df.copy()

# Remove exact duplicate rows
sales = sales.drop_duplicates()

# Convert date and numeric columns to suitable data types
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["UnitPrice"] = pd.to_numeric(sales["UnitPrice"], errors="coerce")

print("Rows after removing duplicates:", len(sales))

In [ ]:
# Count cancellation records before filtering
cancellation_rows = sales["InvoiceNo"].astype(str).str.startswith("C").sum()
print("Cancellation rows:", cancellation_rows)

# Keep completed sales only
sales = sales[
    (~sales["InvoiceNo"].astype(str).str.startswith("C")) &
    (sales["Quantity"] > 0) &
    (sales["UnitPrice"] > 0) &
    (sales["InvoiceDate"].notna())
].copy()

print("Rows after sales filters:", len(sales))

In [ ]:
# CustomerID is missing for some transactions.
# We keep those rows for total sales analysis.
print("Rows without CustomerID:", sales["CustomerID"].isna().sum())

# Fill missing product descriptions with a clear label
sales["Description"] = sales["Description"].fillna("Unknown product")

# Check the cleaned data
sales.isnull().sum()

## 7. Create useful columns (feature engineering)

Feature engineering means creating new columns that help us answer business questions.

- **Revenue** = Quantity × UnitPrice. This is the value of one invoice line.
- **InvoiceMonth** helps us group sales by month.

We are not calculating profit because product costs are not included in the dataset.

In [ ]:
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["InvoiceMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["Year"] = sales["InvoiceDate"].dt.year

sales[["InvoiceNo", "Description", "Quantity", "UnitPrice", "Revenue", "InvoiceMonth"]].head()

## 8. Calculate the main business numbers (KPIs)

A **KPI (Key Performance Indicator)** is a number used to understand business performance.

- Total revenue: sum of all line revenue.
- Total orders: number of unique invoices.
- Units sold: total quantity.
- Average order value (AOV): revenue divided by number of orders.
- Identified customers: unique non-missing customer IDs.

In [ ]:
total_revenue = sales["Revenue"].sum()
total_orders = sales["InvoiceNo"].nunique()
total_units = sales["Quantity"].sum()
identified_customers = sales["CustomerID"].nunique()
average_order_value = total_revenue / total_orders

kpis = pd.DataFrame({
    "Metric": ["Total Revenue (£)", "Total Orders", "Units Sold", "Identified Customers", "Average Order Value (£)"],
    "Value": [round(total_revenue, 2), total_orders, int(total_units), identified_customers, round(average_order_value, 2)]
})
kpis

## 9. Analyse monthly revenue

Grouping lets us combine many transaction rows into a smaller summary. Here we add up revenue for each month.

In [ ]:
monthly_sales = sales.groupby("InvoiceMonth")["Revenue"].sum().reset_index()
monthly_sales = monthly_sales.sort_values("InvoiceMonth")
monthly_sales

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="InvoiceMonth", y="Revenue", marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Find the top-selling products by revenue

We group transactions by product description and add up the revenue. Then we sort from highest to lowest.

In [ ]:
top_products = (
    sales.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_products

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top_products, x="Revenue", y="Description")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 11. Compare revenue by country

This helps us see which countries contribute most to the recorded sales. The dataset's prices are in GBP, so the revenue is shown in pounds.

In [ ]:
top_countries = (
    sales.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_countries

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top_countries, x="Revenue", y="Country")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 12. Find the highest-spending customers

For customer-level analysis, a CustomerID is necessary. Therefore, we exclude rows with a missing CustomerID only in this section. The overall sales KPIs above still include those transactions.

In [ ]:
customer_sales = sales.dropna(subset=["CustomerID"]).copy()

top_customers = (
    customer_sales.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_customers

## 13. Save the cleaned data and results

Saving the cleaned table as a CSV file makes it easy to reuse in Excel, Power BI, or another Python analysis. The notebook will create an `outputs` folder automatically.

In [ ]:
from pathlib import Path

output_folder = Path("outputs")
output_folder.mkdir(exist_ok=True)

sales.to_csv(output_folder / "cleaned_sales.csv", index=False)
monthly_sales.to_csv(output_folder / "monthly_sales.csv", index=False)
top_products.to_csv(output_folder / "top_products.csv", index=False)
top_countries.to_csv(output_folder / "top_countries.csv", index=False)
kpis.to_csv(output_folder / "key_metrics.csv", index=False)

print("Cleaned data and analysis results saved in the outputs folder.")

## 14. Conclusion

Use the tables and charts above to write down what the data actually shows. Do not guess or invent results. Some questions to answer:

1. Which month had the highest revenue?
2. Which product generated the most revenue?
3. Which country ranked first?
4. What was the average order value?
5. What data-quality issues did we find?

### Limitations

- The dataset covers roughly one year and one retailer.
- Revenue is calculated from quantity multiplied by unit price; it is not profit.
- Customer analysis excludes rows without a CustomerID.
- Cancellation and return records are excluded from the completed-sales analysis described in this notebook.
- The dataset does not provide a complete accounting view of tax, shipping, discounts, or product costs.

### Dataset source and attribution

Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Dataset page: https://archive.ics.uci.edu/dataset/352/online+retail  
License: CC BY 4.0. Please retain attribution to the original dataset provider.

---

**Project workflow:** Data loading → Data understanding → Data cleaning → Feature creation → KPI analysis → Visualisation → Business interpretation.